# Импорты

In [1]:
from IPython.display import clear_output     # очистка вывода ячейки (попытка создания непрерывной "анимации")
from datetime import datetime as dt          # получение времени и даты для вставки в название лога
import random as rnd                         # создание "случайностей"
import time                                  # для блокировки потока (попытка фиксировать FPS)

# Вспомогательные функции

In [2]:
def choice_prob(more_prob, less_prob, prob=0.8):
    """Случайно выбирает элемент из объединения двух списков с определённой вероятностью предпочтения элемента из приоритетного списка

    Args:
        more_prob (list[object]): Приоритетный список
        less_prob (list[object]): Второстепенный список
        prob (float): Вероятность выбора случайного элемента из приоритетного (первого) списка

    Returns:
        Случайный элемент из объединения списков
    """
    if more_prob and less_prob:
        weights = [prob / len(more_prob)] * len(more_prob) + [(1 - prob) / len(less_prob)] * len(less_prob)
    elif less_prob:
        weights = [1 / len(less_prob)] * len(less_prob)
    else:
        weights = [1 / len(more_prob)] * len(more_prob)

    return rnd.choices(more_prob + less_prob, weights=weights, k=1)[0]


def die_pred(current, max_stat):
    """Определяет смерть существа

    Args:
        current (int): Текущий показатель
        max (int): Максимальный показатель

    Returns:
        bool: Решение о смерти существа
    """
    if current >= 2 * max_stat:
        return True
    death_chance = (current / max_stat) ** 2
    return rnd.random() < death_chance


def sex_choice():
    """Определяет пол существа

    Returns:
        str: Пол существа
    """
    while True:
        yield "XX"
        yield "XY"


# генератор для чередования пола существ
sex_gen = sex_choice()


# функция-сокращение для просмотра ближайших координат (крест)
crusade = lambda x, y: [(x - 1, y), (x + 1, y), (x, y - 1), (x, y + 1)]

# функция-сокращение для просмотра ближайших координат (круг с радиусом r)
circle = lambda x, y, r: [(i, j) for i in range(x - r, x + r + 1) for j in range(y - r, y + r + 1) if (i, j) != (x, y)]

# функция-сокращение для поиска названия базового класса
find_parent_name = lambda obj: obj.__class__.__bases__[0].__name__

# функция-сокращение для поиска названия класса
find_class_name = lambda obj: obj.__class__.__name__

# Базовые классы

In [3]:
class CelestialObj:
    """Небесное тело, от которого идёт счёт времени в мире"""
    def __init__(self):
        """Инициализирует объект от нулевой точки времени"""
        self.ticks = 0
        self.time = "Day 0 | 0:00 (Night)"
    

    def tick(self):
        """Совершает шаг во времени"""
        self.ticks += 1
        self.time = f"Day {self._get_day()} | {self._get_time()}:00 ({self._get_pos()})"


    def _get_day(self):
        """Вычисляет день
        
        Returns:
            int: Порядковое значение дня
        """
        return self.ticks // 24


    def _get_time(self):
        """Вычисляет время
        
        Returns:
            int: Часы в 24-часовом формате"""
        return self.ticks % 24
    

    def _get_pos(self):
        """Вычисляет временную метку
        
        Returns:
            str: Словесное представление промежутка времени"""
        return ["Night", "Morning", "Day", "Afternoon"][((self.ticks % 24 + 1) // 6) % 4]

In [4]:
class World():
    """Мир, представленный плоскостью с набором сущностей
    
    Attributes:
        width (int): Ширина плоскости мира
        height (int): Высота плоскости мира
        cel_obj (CelestialObj): Небесное тело от которого идёт времяисчисление
        entities (dict[tuple[int, int]: object]): Сущности мира в формате {координата: сущность}
        plain (list[list[str]]): Плоскость мира
        free_places (list[tuple[int, int]]): Свободные координаты
    """
    def __init__(self, size, cel_obj=None):
        """Инициализирует пустой мир
        
        Args:
            size (tuple[int, int]): Размер мира в формате (ширина, высота)
            cel_obj (CelestialObj): Небесное тело
        """
        if cel_obj is None:
            cel_obj = CelestialObj()
            
        self.width = size[0]
        self.height = size[1]
        self.cel_obj = cel_obj
        self.entities = {}
        self.plain = [["|   " for _ in range(self.width + 1)] for _ in range(self.height)]
        self.free_places = [(x, y) for x in range(self.width) for y in range(self.height)]
        

    def add_entity(self, Entity):
        """Добавляет сущность на плоскость мира
        
        Args:
            Entity (object): Сущность
        """
        self.entities[Entity.coord] = Entity
        self.free_places.remove(Entity.coord)
        x, y = Entity.coord
        self.plain[y][x] = Entity.mark
    

    def del_entity(self, Entity):
        """Удаляет сущность с плоскости мира
        
        Args:
            Entity (object): Сущность
        """
        self.entities.pop(Entity.coord)
        self.free_places.append(Entity.coord)
        x, y = Entity.coord
        self.plain[y][x] = "|   "

    
    def repl_entity(self, New_entity):
        """Заменяет сущность на плоскости мира
        
        Args:
            New_entity (object): Сущность
        """
        self.entities[New_entity.coord] = New_entity
        x, y = New_entity.coord
        self.plain[y][x] = New_entity.mark

    
    def move_entity(self, Entity, new_place):
        """Меремещает сущность на плоскости мира
        
        Args:
            Entity (object): Сущность
            new_place (tuple[int, int]): Координата нового расположения
        """
        self.entities.pop(Entity.coord)
        self.free_places.append(Entity.coord)
        x, y = Entity.coord
        self.plain[y][x] = "|   "
        self.entities[new_place] = Entity

        if new_place in self.free_places:
            self.free_places.remove(new_place)
            
        x, y = new_place
        self.plain[y][x] = Entity.mark


    def __contains__(self, obj):
        """Определяет есть ли объект в мире
        
        Args:
            obj (tuple[int, int] | object): Объект

        Returns:
            bool: Содержится ли объект в мире
        """
        if isinstance(obj, tuple):
            return 0 <= obj[0] < self.width and 0 <= obj[1] < self.height
        return obj in self.entities.values()

In [5]:
class GameLoop:
    """Цикл симуляции, упрощающий взаимодействие классов
   
    Attributes:
        world (World): Мир
    """
    def __init__(self, entities_list=[], world_size=(10, 10)):
        """Инициализирует игровой цикл и сущности на плоскость мира
    
        Args:
            entities_list (list[tuple[object, int]]): Список классов сущностей и их количетсво, расположенное в мире
            world_size (tuple[int, int]): Размер мира
        """
        self.world = World(world_size)

        for Entity, num in entities_list:
            for _ in range(num):
                place = rnd.choice(self.world.free_places)
                self.world.add_entity(Entity(self.world, place))


        print(self._render_frame())


    def _tick(self):
        """Совершает шаг цикла"""
        self.world.cel_obj.tick()

        entities = list(self.world.entities.values())
        rnd.shuffle(entities)

        for Entity in entities:
            Entity.live()


    def _render_frame(self):
        """Комплектует визуальный кадр состояния объектов
    
        Returns:
            str: Кадр
        """
        time_state = self.world.cel_obj.time
        borders = "...." * self.world.width + "."
        plain = "\n".join(["".join(row) for row in self.world.plain])
        frame = "\n".join([time_state, borders, plain, borders])
        return frame

    
    def _log(self, frame, file):
        """Логирует кадр
    
        Args:
            frame (str): Кадр
            file (str): Название файла для записи лога
        """
        with open(f"{file}.txt", "a", encoding="utf-8") as file:
            file.write(frame + "\n")


    def loop(self, ticks=24, delay=0.2, logging=False, log_filename=None, clickable=False):
        """Воспроизводит цикл симуляции с визуализацией
    
        Args:
            ticks (int): Количество шагов (оно же количество кадров и часов)
            delay (float): Задержка перед выводом следующего кадра
            loging (bool): Значение определяющее наличие логирования у запуска цикла
            log_filename (str): Название файла для размещения логов
            clickable (bool): Смена кадра по клику
        """
        if log_filename is None:
            log_filename = dt.now().strftime(r"%H-%M-%S_%d-%m-%Y")

        for _ in range(ticks):
            clear_output(wait=True)

            frame = self._render_frame()
            if logging: self._log(frame, log_filename)
            print(frame)
            self._tick()

            time.sleep(delay)
            if clickable:
                input()

# Сущности

In [6]:
class Entity:
    """Базовый класс для любых сущностей, располагаемых в мире
    
    Attributes:
        world (World): Мир в котором расположена сущность
        coord (tuple[int, int]): Координата на которой располагается сущность на плоскости мира
        mark (str): Метка сущности на визуализированной плоскости
        active_time (list[str]): Список меток активного времени для сущности
        age (int): Возраст сущности
        max_age (int): Максимальный возраст сущности
    """
    def __init__(self, world, coord):
        """Инициализирует сущность

        Args:
            world (World): Мир в котором расположена сущность
            coord (tuple[int, int]): Координата на которой располагается сущность на плоскости мира
        """
        self.world = world
        self.coord = coord
        self.mark = ""
        self.active_time = []
        self.age = 0
        self.max_age = 0


    def _do_nothing(self):
        """Ничего не делает"""
        pass


    def _die(self):
        """Убирает сущность из мира"""
        self.world.del_entity(self)

    
    def _adapt(self):
        """Ничего не делает, потенциально переопределяется в наследниках"""
        pass


    def _lifecycle(self):
        """Ничего не делает, потенциально переопределяется во время выполнения self._adapt()"""
        pass


    def live(self):
        """Совершает жизненный цикл"""
        self._adapt()
        self._lifecycle()

## Растения

In [7]:
class Plant(Entity):
    """Базовый класс для всех растений

    Attributes:
        world (World): Мир в котором расположено растение
        coord (tuple[int, int]): Координата на которой расположено растение на плоскости мира
        mark (str): Метка растения на визуализированной плоскости
        active_time (list[str]): Список меток активного времени для растения
        age (int): Возраст растения
        max_age (int): Максимальный возраст растения
        grow_prob (int): Вероятность с которой растение будет распространяться в свободных направлениях
        repl_prob (int): Вероятность с которой растение вымещает другое растение
    """
    def __init__(self, world, coord):
        """Инициализирует растение

        Args:
            world (World): Мир в котором расположено растение
            coord (tuple[int, int]): Координата по которой расположено растение на плоскости
        """
        super().__init__(world, coord)
        self.grow_prob = 0
        self.repl_prob = 0


    def _adapt(self):
        """Меняет стратегию жизненного цикла растения в зависимости от состояния других объектов"""
        die = die_pred(self.age, self.max_age)
        self.age += 1
        
        if not (self.world.cel_obj._get_pos() in self.active_time):
            self._lifecycle = self._die if die else self._do_nothing
        elif die:
            self._lifecycle = self._die
        else:
            self._lifecycle = self._grow
    

    def _grow(self):
        """Создаёт новое растение в соседней координате"""
        if rnd.randint(0, 100) > self.grow_prob:
            return

        coords = crusade(*self.coord)

        places = []
        prob_places = []

        for coord in coords:
            if not coord in self.world:
                continue

            if coord in self.world.free_places:
                places.append(coord)
                continue
                
            entity = self.world.entities[coord]
            same_parent = find_parent_name(entity) == find_parent_name(self)
            diff_class = find_class_name(entity) != find_class_name(self)
            
            if same_parent and diff_class:
                prob_places.append(coord)

        if places and prob_places:
            place = choice_prob(places, prob_places, self.repl_prob / 100)
        elif places:
            place = rnd.choice(places)
        elif prob_places and rnd.randint(0, 100) < self.repl_prob:
            place = rnd.choice(prob_places)
        else:
            return
        
        new_plant = self.__class__(self.world, place)

        if place in self.world.free_places:
            self.world.add_entity(new_plant)
        else:
            self.world.repl_entity(new_plant)

In [8]:
class Demi(Plant):
    """Класс сумрачного растения Demi
    
    Attributes:
        world (World): Мир в котором расположено растение
        coord (tuple[int, int]): Координата на которой расположено растение на плоскости мира
        mark (str): Метка растения на визуализированной плоскости
        active_time (list[str]): Список меток активного времени для растения
        age (int): Возраст растения
        max_age (int): Максимальный возраст растения
        grow_prob (int): Вероятность с которой растение будет распространяться в свободных направлениях
        repl_prob (int): Вероятность с которой растение вымещает другое растение
    """
    def __init__(self, world, coord):
        """Инициализирует растение Demi

        Args:
            world (World): Мир в котором расположено растение
            coord (tuple[int, int]): Координата по которой расположено растение на плоскости
        """
        super().__init__(world, coord)
        self.mark = "|\033[48;2;93;121;48;1m D \033[0m"
        self.active_time = ["Morning", "Afternoon"]
        self.max_age = 48
        self.grow_prob = 60
        self.repl_prob = 3

In [9]:
class Obscurite(Plant):
    """Класс ночного растения Obscurite
    
    Attributes:
        world (World): Мир в котором расположено растение
        coord (tuple[int, int]): Координата на которой расположено растение на плоскости мира
        mark (str): Метка растения на визуализированной плоскости
        active_time (list[str]): Список меток активного времени для растения
        age (int): Возраст растения
        max_age (int): Максимальный возраст растения
        grow_prob (int): Вероятность с которой растение будет распространяться в свободных направлениях
        repl_prob (int): Вероятность с которой растение вымещает другое растение
    """
    def __init__(self, world, coord):
        """Инициализирует растение Demi

        Args:
            world (World): Мир в котором расположено растение
            coord (tuple[int, int]): Координата по которой расположено растение на плоскости
        """
        super().__init__(world, coord)
        self.mark = "|\033[48;5;58m O \033[0m"
        self.active_time = ["Night", "Afternoon"]
        self.max_age = 48
        self.grow_prob = 60
        self.repl_prob = 3

In [10]:
class Lumiere(Plant):
    """Класс дневного растения Lumiere
    
    Attributes:
        world (World): Мир в котором расположено растение
        coord (tuple[int, int]): Координата на которой расположено растение на плоскости мира
        mark (str): Метка растения на визуализированной плоскости
        active_time (list[str]): Список меток активного времени для растения
        age (int): Возраст растения
        max_age (int): Максимальный возраст растения
        grow_prob (int): Вероятность с которой растение будет распространяться в свободных направлениях
        repl_prob (int): Вероятность с которой растение вымещает другое растение
    """
    def __init__(self, world, coord):
        """Инициализирует растение Demi

        Args:
            world (World): Мир в котором расположено растение
            coord (tuple[int, int]): Координата по которой расположено растение на плоскости
        """
        super().__init__(world, coord)
        self.mark = "|\033[48;2;78;154;64;1m L \033[0m"
        self.active_time = ["Morning", "Day"]
        self.max_age = 48
        self.grow_prob = 60
        self.repl_prob = 3

## Животные

In [12]:
class Animal(Entity):
    """Базовый класс для всех животных
    
    Attributes:
        world (World): Мир в котором расположено животное
        coord (tuple[int, int]): Координата на которой располагается животное на плоскости мира
        mark (str): Метка животного на визуализированной плоскости
        active_time (list[str]): Список меток активного времени для животного
        age (int): Возраст животного
        max_age (int): Максимальный возраст животного
        swarm (list[tuple[int, int]]): Координаты стаи с которой связано животное
        diet (list[str]): Рацион питания животного
        sex (str): Пол животного
        aggr (int): Агорессивность животного
        hunger (int): Уровень голода животного
        max_hunger (int): Максимальный уровень голода животного
        max_swarm (int): Максимальный размер стаи
    """
    def __init__(self, world, coord):
        """Инициализирует животное

        Args:
            world (World): Мир в котором расположено животное
            coord (tuple[int, int]): Координата по которой расположено животное на плоскости
        """
        super().__init__(world, coord)
        self.swarm = [coord]
        self.diet = []
        self.sex = next(sex_gen)
        self.aggr = 0
        self.hunger = 0
        self.max_hunger = 0
        self.max_swarm = 0


    def _adapt(self):
        """Меняет стратегию жизненного цикла животного в зависимости от состояния других объектов"""
        if self.world.cel_obj.ticks == 1:
            self._lifecycle = self._repr
            return
        
        self.swarm = self._find_swarm()
        self.aggr = len(self.swarm)
        self.age += 1

        age = die_pred(self.age, self.max_age)
        hunger = die_pred(self.hunger, self.max_hunger)
        die = max(age, hunger)

        if not (self.world.cel_obj._get_pos() in self.active_time):
            self._lifecycle = self._do_nothing
        elif die:
            self._lifecycle = self._die
        else:
            if self.hunger < self.max_hunger * 2 // 3 and self._find_pair() and self.sex == "XX" and self.age > 12:
                action = self._repr
                self.hunger = self.max_hunger // 3
            elif self.hunger > (self.max_hunger // 10) or self.aggr >= self.max_swarm:
                action = self._eat
            else:
                action = self._move_to_swarm

            def lifecycle():
                action()
                self.hunger += 1

            self._lifecycle = lifecycle


    def _eat(self):
        """Совершает процедуру питания животного"""
        target = self._hunt()

        if target is None:
            self._move()
        else:
            self.world.del_entity(self.world.entities[target])
            self._move(target)
            self.hunger = 0


    def _hunt(self):
        """Совершает процедуру охоты животного
        
        Returns:
            tuple: Координата жертвы
        """
        if self.aggr > self.max_swarm:
            self.swarm.remove(self.coord)
            return rnd.choice(self.swarm)

        observed = circle(*self.coord, r=2)
        rnd.shuffle(observed)

        for coord in observed:
            if coord in self.world.entities.keys():
                target_class = find_class_name(self.world.entities[coord])
                if target_class in self.diet:
                    return coord

        return None
    

    def _move(self, target=None):
        """Совершает процедуру перемещения животного
        
        Args:
            target (tuple[int, int]): Цель перемещения
            """
        if target is None:
            coords = crusade(*self.coord)
            places = [self.coord]

            for coord in coords:
                if not coord in self.world:
                    continue
                elif coord in self.world.free_places or find_parent_name(self.world.entities[coord]) != find_parent_name(self):
                    places.append(coord)

            target = rnd.choice(places)

        self.world.move_entity(self, target)
        self.coord = target


    def _move_to_swarm(self):
        """Совершает процедуру перемещения животного в пределах стаи"""
        target = rnd.choice(self.swarm)
        places = circle(*target, r=2)
        rnd.shuffle(places)

        for place in places:
            if place in self.world.free_places:
                self._move(place)
                return
        
        self._move()


    def _find_pair(self):
        """Предикат определения наличия пары для продолжения рода
        
        Returns:
            bool: Наличие пары у животного
        """
        for coord in self.swarm:
            if self.world.entities[coord].sex != self.sex:
                return True
        
        return False


    def _repr(self):
        """Создаёт новое животное в соседней координате"""
        coords = crusade(*self.coord)
        place = None

        for coord in coords:
            if not coord in self.world:
                continue
            elif coord in self.world.free_places or find_parent_name(self.world.entities[coord]) != find_parent_name(self):
                place = coord
                break

        if place is None:
            return
        
        new_animal = self.__class__(self.world, place)
        if place in self.world.free_places:
            self.world.add_entity(new_animal)
        else:
            self.world.repl_entity(new_animal)


    def _find_swarm(self):
        """Собирает стаю
        
        Returns:
            list[tuple[int]]: Координаты животных из стаи
        """
        swarm = [self.coord]
        visited = set()
        queue = [self.coord]
        visited.add(self.coord)

        while queue:
            current_coord = queue.pop(0)
            neighbors = circle(*current_coord, r=2)

            for neighbor in neighbors:
                if neighbor not in visited:
                    visited.add(neighbor)
                    if (neighbor in self.world.entities and 
                        find_class_name(self.world.entities[neighbor]) == find_class_name(self)):
                        swarm.append(neighbor)
                        queue.append(neighbor)

        return swarm

In [13]:
class Pauvre(Animal):
    """Класс травоядного животного Pauvre
    
    Attributes:
        world (World): Мир в котором расположено животное
        coord (tuple[int, int]): Координата на которой располагается животное на плоскости мира
        mark (str): Метка животного на визуализированной плоскости
        active_time (list[str]): Список меток активного времени для животного
        age (int): Возраст животного
        max_age (int): Максимальный возраст животного
        swarm (list[tuple[int, int]]): Координаты стаи с которой связано животное
        diet (list[str]): Рацион питания животного
        sex (str): Пол животного
        aggr (int): Агорессивность животного
        hunger (int): Уровень голода животного
        max_hunger (int): Максимальный уровень голода животного
        max_swarm (int): Максимальный размер стаи
    """
    def __init__(self, world, coord):
        """Инициализирует животное

        Args:
            world (World): Мир в котором расположено животное
            coord (tuple[int, int]): Координата по которой расположено животное на плоскости
        """
        super().__init__(world, coord)
        self.mark = "|\033[48;5;1m P \033[0m"
        self.active_time = ["Morning", "Day", "Afternoon"]
        self.max_age = 480
        self.swarm = [coord]
        self.diet = ["Lumiere"]
        self.max_hunger = 150
        self.max_swarm = 5

In [14]:
class Malheureux(Animal):
    """Класс всеядного животного Malheureux
    
    Attributes:
        world (World): Мир в котором расположено животное
        coord (tuple[int, int]): Координата на которой располагается животное на плоскости мира
        mark (str): Метка животного на визуализированной плоскости
        active_time (list[str]): Список меток активного времени для животного
        age (int): Возраст животного
        max_age (int): Максимальный возраст животного
        swarm (list[tuple[int, int]]): Координаты стаи с которой связано животное
        diet (list[str]): Рацион питания животного
        sex (str): Пол животного
        aggr (int): Агорессивность животного
        hunger (int): Уровень голода животного
        max_hunger (int): Максимальный уровень голода животного
        max_swarm (int): Максимальный размер стаи
    """
    def __init__(self, world, coord):
        """Инициализирует животное

        Args:
            world (World): Мир в котором расположено животное
            coord (tuple[int, int]): Координата по которой расположено животное на плоскости
        """
        super().__init__(world, coord)
        self.mark = "|\033[48;2;128;0;32;1m M \033[0m"
        self.active_time = ["Morning", "Afternoon"]
        self.max_age = 480
        self.swarm = [coord]
        self.diet = ["Demi", "Obscurite", "Pauvre"]
        self.max_hunger = 150
        self.max_swarm = 7

# Тест

In [15]:
game = GameLoop([(Demi, 4), (Obscurite, 4), (Lumiere, 4), (Malheureux, 2), (Pauvre, 2)], (20, 16))
game.loop(100, logging=True, log_filename="all")

Day 4 | 3:00 (Night)
.................................................................................
| L |   | L |   | L | L | L | L | L |   |   | L | L | D |   | D | D | D | D | D |   
| L | L |   | L |   |   |   |   | L |   |   |   | L | L | D | D | D | D | D | M |   
| L |   |   |   |   | L | L |   | L |   | P |   | L |   |   | D |   |   |   | D |   
|   | L |   |   | L |   |   |   | L | L |   |   |   | L | D | D | D | D | D | D |   
|   |   | L | L | L |   | L |   |   |   | L |   |   |   | L | D | D | D | D | D |   
| D | D | L | L | L |   |   |   |   |   | L |   |   | L |   | D | D | D | M |   |   
|   | D |   | L |   | L |   | L | L |   | L | L | L |   |   | D |   | D |   |   |   
| D | D | D | L |   |   | L | D |   | L |   |   |   |   | L | D | D |   | D | D |   
| D | D | D | D | D |   | D | D |   | L | L | L | L | L |   | D |   | D | D | D |   
| D | D |   |   | D | D | D |   |   | D |   |   |   | L | D | D | D | D | D | D |   
| D | D |   | D | D | D | D | D | D |   | D | D

# Не реализованые фичи:
- Параметр `verbosity` - "разговорчивость" вывода (1 - оповещение о росте/размножении/питании/умирании, 0 - только картинка)
- Полноценная работа стай